### Data Loading and Preprocessing

Imports

In [1]:
import os
import matplotlib.pyplot as plt 
import math
from collections import Counter, defaultdict

Set Path and Load Data

In [5]:
data_path = "data/fr_train.conllu"

with open(data_path, "r", encoding = "utf-8") as f:
    lines = f.readlines()
    
len(lines)

407728

Verify Data

In [6]:
first_twenty = lines[:20]

for l in first_twenty:
    print(l)

# global.columns = ID FORM LEMMA UPOS XPOS FEATS HEAD DEPREL DEPS MISC

# sent_id = fr-ud-train_00001

# text = Les commotions cérébrales sont devenu si courantes dans ce sport qu'on les considére presque comme la routine.

1	Les	le	DET	_	Definite=Def|Number=Plur|PronType=Art	2	det	_	wordform=les

2	commotions	commotion	NOUN	_	Number=Plur	5	nsubj	_	Gender[lex]=Fem

3	cérébrales	cérébral	ADJ	_	Gender=Fem|Number=Plur	2	amod	_	_

4	sont	être	AUX	_	Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin	5	aux:tense	_	_

5	devenu	devenir	VERB	_	Gender=Masc|Number=Sing|Typo=Yes|VerbForm=Part|Voice=Act	0	root	_	CorrectForm=devenues|CorrectGender=Fem|CorrectNumber=Plur|Tense[denom]=Past

6	si	si	ADV	_	_	7	advmod	_	_

7	courantes	courant	ADJ	_	Gender=Fem|Number=Plur	5	xcomp	_	_

8	dans	dans	ADP	_	_	10	case	_	_

9	ce	ce	DET	_	Gender=Masc|Number=Sing|PronType=Dem	10	det	_	_

10	sport	sport	NOUN	_	Number=Sing	7	obl:mod	_	Gender[lex]=Masc

11	qu'	que	SCONJ	_	_	14	mark	_	SpaceAfter=No

12	on	on	PRON	_

Extract Tokens and POS Tags

In [ ]:
tokens = [] # List to store tokens
tags = [] # List to store POS tags
sentences = [] # List to store sentences
tag_sequences = [] # List to store tag sequences for each sentence
current_sentence = [] # Temporary list to build current sentence
current_tags = [] # Temporary list to build current tags

# Iterate through each line in the file
for line in lines:
    line = line.strip() # Remove leading and trailing whitespace
    # Check if line is empty, indicating end of sentence
    if not line:
        # Check if current sentence is not empty
        if current_sentence:
            sentences.append(current_sentence) # Add current sentence to sentence list
            tag_sequences.append(current_tags) # Add current tags to tag list
            current_sentence = [] # Reset current sentence
            current_tags = [] # Reset current tags
        continue
    
    if line.startswith("#"):
        continue # Skip comment lines

    parts = line.split("\t") # Split line into parts using tab as delimiter
    
    if "-" in parts[0]:
        continue # Skip multi-word token lines

    word = parts[1].lower() # Get word and convert to lowercase
    tag = parts[3] # Get UD POS tag
    tokens.append(word) # Add word to tokens list
    tags.append(tag) # Add tag to tags list
    current_sentence.append(word) # Add word to current sentence
    current_tags.append(tag) # Add tag to current tags

len(tokens), len(sentences)

(354648, 14450)

Add Sentence Boundaries

In [9]:
processed_sentences = [["<s>"] + sent + ["</s"] for sent in sentences] # Add start and end tokens to sentences
processed_tags = [["<s>"] + ts + ["/s>"] for ts in tag_sequences] # Add start and end tokens to tag sequences

Flatten Tokens

In [10]:
flat_tokens = [w for s in processed_sentences for w in s] # Flatten sentence list into single token list
flat_tags = [t for ts in processed_tags for t in ts] # Flatten tag list into single tag list

Analyze Vocabulary

In [11]:
vocab = Counter(flat_tokens) # Create vocabulary counter for tokens
tagset = set(flat_tags) # Create set of unique tags

print(f"Vocabulary Size: {len(vocab)}")
print(f"Number of POS Tags: {len(tagset)}")

most_common = vocab.most_common(20) # Get 20 most common tokens

# Iterate through most common tokens and print them with their counts
for w, c in most_common:
    print(f"{w}: {c}")

Vocabulary Size: 39685
Number of POS Tags: 18
de: 23926
,: 16599
<s>: 14450
</s: 14450
le: 13924
.: 13366
la: 9830
à: 9280
les: 8706
et: 7213
l': 6539
en: 5479
est: 4738
d': 4518
un: 3881
une: 3339
il: 3213
dans: 2779
pour: 2241
(: 2211


### Hidden Markov Model POS Tagging

Transition Probabilities

In [13]:
# Count Transitions with Nested Dictionary: P(tag_i | tag_{i-1})
transitions = defaultdict(lambda: defaultdict(int))

# Iterate through tag sequences
for ts in processed_tags:
    # Iterate through tag pairs in the sequence
    for i in range(len(ts) - 1):
        t1, t2 = ts[i], ts[i + 1] # Get current and next tag
        transitions[t1][t2] += 1 # Increment transition count

# Convert counts to probabilities
transition_probs = {} # Dictionary to store transition probabilites

# Iterate through each tag in transitions
for t1 in transitions:
    total = sum(transitions[t1].values()) # Get total transitions from tag t1
    transition_probs[t1] = {t2: count / total for t2, count in transitions[t1].items()} # Normalize counts to probabilities

Compute Emission Probabilites

In [14]:
emissions = defaultdict(lambda: defaultdict(int)) # Nested dictionary to count emissions of words given tags

# Iterate through sentences and their corresponding tag sequences
for sent, ts in zip(processed_sentences, processed_tags):
    # Iterate through words and tags in the sentence
    for w, t in zip(sent, ts):
        emissions[t][w] += 1 # Count emissions of word w given tag t

# Convert counts to probabilities
emission_probs = {} # Dictionary to store emission probabilities

# Iterate through each tag in emissions
for t in emissions:
    total = sum(emissions[t].values()) # Get total emissions for tag t
    emission_probs[t] = {w: count / total for w, count in emissions[t].items()} # Normalize counts to probablilities